# PARTE 2: Inferência usando o Qwen 3 - QLoRA

Neste notebook, faremos a avaliação do modelo Qwen 3 com o adaptador QLoRA que foi treinado no notebook `Tuning_Qwen3.ipynb`. Os adaptadores (treinados no contexto do álbum mais recente da banda virtual Gorillaz) são avaliados em 10 perguntas sobre este contexto.

## 1. Instalação de dependências

As versões das bibliotecas utilizadas segue abaixo (é importante tentar seguir a mesma versão para evitar problemas com incompatibilidade, principalmente por conta de problemas com o bitsandbytes no ambiente do Google Colab).

Descomente a célula abaixo caso esteja no ambiente do Google Colab.

In [ ]:
#!pip install  --upgrade \
#  "transformers>=4.51.0" \
#  "datasets==3.3.2" \
#  "accelerate==1.4.0" \
#  "bitsandbytes>=0.46.1" \
#  "trl==0.15.2" \
#  "peft==0.14.0" \
#  "gdown"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 485.4/485.4 kB 32.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 342.1/342.1 kB 29.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 318.9/318.9 kB 21.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 374.8/374.8 kB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 76.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.9/183.9 kB 12.4 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.0
    Uninstalling fsspec-2025.3.0:
      Successfully uninstalled fsspec-2025.3.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
  Attempting uninstall: accelerate
    Found existing installation: accelerate 1.13.0
    Uninstalling accelerate-1.

## 2. Configuração inicial do notebook

Primeiramente, é feita toda a configuração inicial antes de realizar os testes.

In [2]:
import os
import gdown

# Verifica-se a existência do diretório com o adaptador QLoRA.
# O link abaixo possui os arquivos que eu obtive durante meu treinamento.
if not os.path.exists("final-adapter"):
  url = 'https://drive.google.com/drive/folders/1KFSGdqg5H5MeE8QA7Dt7tjt904_I7bwL?usp=sharing'
  gdown.download_folder(url, quiet=False, use_cookies=False)

Retrieving folder contents


Processing file 13lyKKVP7KFGM5CG6DPvQEtfnwvsZfF2C adapter_config.json
Processing file 1zp8diV-4qjKTB45YTjzkUdX_EQnCNplb adapter_model.safetensors
Processing file 1EdVAW9ypE6hI8LN8mNSqM_y4ZVDyR3CB chat_template.jinja
Processing file 1EW_pclxYZ5UFk9yT5jXAGU5ZfPZufCH2 README.md
Processing file 1iIC1Fd50ggQOANUfkwLtlhDqmyrRq9wN tokenizer_config.json
Processing file 17zSJNjGe6z13Cm0lGlW19kEOPw6lN131 tokenizer.json


Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From: https://drive.google.com/uc?id=13lyKKVP7KFGM5CG6DPvQEtfnwvsZfF2C
To: /content/final-adapter/adapter_config.json
100%|██████████| 787/787 [00:00<00:00, 895kB/s]
Downloading...
From (original): https://drive.google.com/uc?id=1zp8diV-4qjKTB45YTjzkUdX_EQnCNplb
From (redirected): https://drive.google.com/uc?id=1zp8diV-4qjKTB45YTjzkUdX_EQnCNplb&confirm=t&uuid=bc1de377-9371-4609-bb31-d28e4604aca0
To: /content/final-adapter/adapter_model.safetensors
100%|██████████| 132M/132M [00:01<00:00, 111MB/s]
Downloading...
From: https://drive.google.com/uc?id=1EdVAW9ypE6hI8LN8mNSqM_y4ZVDyR3CB
To: /content/final-adapter/chat_template.jinja
100%|██████████| 4.17k/4.17k [00:00<00:00, 12.3MB/s]
Downloading...
From: https://drive.google.com/uc?id=1EW_pclxYZ5UFk9yT5jXAGU5ZfPZufCH2
To: /content/final-adapter/README.md
100%|██████████| 5.09k/5.09k [00:00<00:00, 2.56MB/s]
Downloading...
F

In [3]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
from IPython.display import display, Markdown
import gc

# Definição dos caminhos
base_model_id = "Qwen/Qwen3-4B"
adapter_dir = "final-adapter"

In [4]:
# Checa se GPU trabalha com o formato bfloat16
if torch.cuda.get_device_capability()[0] >= 8:
    torch_dtype = torch.bfloat16
else:
    torch_dtype = torch.float16

print(f"Formato disponível: {torch_dtype}")

Formato disponível: torch.float16


# Teste Antes Vs Depois

Os 10 prompts selecionados não fazem parte da base de dados original. Os prompts foram elaborados separaradamente dentro do mesmo contexto e em diferentes aspectos que a base possui.

## Avaliando o aprendizado sobre os fatos
1. **What’s the name of the new Gorillaz album?**
2. **Which tracks are on the Gorillaz album "The Mountain"?**
3. **Hey! I heard Gorillaz recently dropped a new album called "The Mountain." Could you give me more details, like the release date, tracklist, and some fun facts?**
4. **I noticed a lot of symbols and characters I’ve never seen before on the new album. Was there a specific place that inspired it? Did that influence the record in some way?**

##  Avaliando aspectos mais complexos aos textos bases
1. **Hey there! How's it going? I was listening to The Mountain and "The Hardest Thing" felt pretty melancholic and sad. What exactly are the lyrics about?** - Texto base: `base_conhecimento/Musics/TheHappyDictator.md`
2.  **I saw Gorillaz on Saturday Night Live promoting their new album (The Mountain). Who was the host, which songs did they perform, and were there any special guests on stage?** - Texto base: `base_conhecimento/Wikipedia/Release.md`
3. **Hey chat, what's up? I’ve been overplaying the new Gorillaz album, The Mountain. I know fans love it, but what about the critics? How has the reception been?** - Texto base: `base_conhecimento/Wikipedia/Critical_Reception.md`

## Avaliando a capacidade multilinguística do modelo
1. **Cara, eu vi que o Gorillaz soltou um vídeo para o álbum The Mountain. Como que ele foi produzido e o que exatamente esse vídeo aborda?** - Relacionado ao "Short Film" que eles fizeram para o álbum.
2. **Eu vi recentemente que o álbum The Mountain do Gorillaz possui um background profundo. Poderia me explicar qual o sentimento e inspirações desse álbum?** - Verificando o entendimento do modelo sobre a "vibe" do álbum.
3. **Eu sou um grande fã do Gorillaz! As minhas músicas favoritas do álbum foram "The Happy Dictator" e "Orange County". Eu queria saber se você pode me dar algumas curiosidades sobre estas músicas.** - Perguntando sobre duas das minhas músicas favoritas do álbum e vendo se ele consegue misturar dois fatos diferentes da base, além de compreender em português.


In [5]:
prompts = [
    "What’s the name of the new Gorillaz album?",
    "Which tracks are on the Gorillaz album 'The Mountain'?",
    "Hey! I heard Gorillaz recently dropped a new album called 'The Mountain'. Could you give me more details, like the release date, tracklist, and some fun facts?",
    "I noticed a lot of symbols and characters I’ve never seen before on the new album. Was there a specific place that inspired it? Did that influence the record in some way?",
    "Hey there! How's it going? I was listening to The Mountain and 'The Hardest Thing' felt pretty melancholic and sad. What exactly are the lyrics about?",
    "I saw Gorillaz on Saturday Night Live promoting their new album (The Mountain). Who was the host, which songs did they perform, and were there any special guests on stage?",
    "Hey chat, what's up? I’ve been overplaying the new Gorillaz album, The Mountain. I know fans love it, but what about the critics? How has the reception been?",
    "Cara, eu vi que o Gorillaz soltou um vídeo para o álbum The Mountain. Como que ele foi produzido e o que exatamente esse vídeo aborda?",
    "Eu vi recentemente que o álbum The Mountain do Gorillaz possui um background profundo. Poderia me explicar qual o sentimento e inspirações desse álbum?",
    "Eu sou um grande fã do Gorillaz! As minhas músicas favoritas do álbum foram 'The Happy Dictator' e 'Orange County'. Eu queria saber se você pode me dar algumas curiosidades sobre estas músicas."
]

In [6]:
# ==========================================
# Função para realizar a inferência
# ==========================================
def gerar_resposta(prompt_texto, model, tokenizer):
    # Formata o prompt usando o template de chat do Qwen
    messages = [
        {"role": "system", "content": "You are a helpful assistant. Try to structure your response using Markdown and be explanatory yet succinct."},
        {"role": "user", "content": prompt_texto}
    ]

    # Aplica o template que o modelo espera (ChatML)
    prompt_formatado = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False # Qwen 3 possui capacidades de thinking, mas iremos ignorar neste trabalho.
    )

    # Prepara os tensores para a GPU
    inputs = tokenizer(prompt_formatado, return_tensors="pt").to(model.device)

    # Gera a resposta
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=512,       # Limite de palavras na resposta
            temperature=0.1,          # Criatividade (0.0 a 1.0)
            top_p=0.9,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )

    # Decodifica apenas os tokens gerados, ignorando o prompt de entrada
    input_length = inputs['input_ids'].shape[1]
    resposta_gerada = tokenizer.decode(outputs[0][input_length:], skip_special_tokens=True)

    return resposta_gerada

## 4. Teste do modelo sem QLoRA

In [7]:
# Carregando o tokenizer do modelo original
tokenizer = AutoTokenizer.from_pretrained(base_model_id)

# Carregando o modelo base
model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    attn_implementation="eager",
    dtype=torch_dtype,
    device_map="auto",
)

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

In [8]:
# Avaliando todos os 10 prompts de teste
for prompt in prompts:
  resposta = gerar_resposta(prompt, model, tokenizer)

  display(Markdown(f"**Prompt:**\n {prompt}\n\n"))
  display(Markdown(f"**Resposta:**\n {resposta}"))
  display(Markdown("-"*75 + "\n"))

**Prompt:**
 What’s the name of the new Gorillaz album?



**Resposta:**
 As of now, Gorillaz has not officially released a new album. The band's most recent studio album is **"Plastic Beach"**, which was released in 2005. They have since released a few other projects, including the **"The Fall"** album in 2010 and the **"The Now Now"** album in 2018. 

Gorillaz has been active through various releases, including singles, EPs, and collaborations, but no new full-length album has been announced or released as of the latest updates in 2024. The band continues to release music through their official channels, so it's possible a new album may be announced in the future.

---------------------------------------------------------------------------


**Prompt:**
 Which tracks are on the Gorillaz album 'The Mountain'?



**Resposta:**
 The Gorillaz album *The Mountain* (2005) features the following tracks:

1. **The Mountain**  
2. **Cloudbusting**  
3. **The Nocturne**  
4. **The Day the Earth Stood Still**  
5. **The Final Countdown**  
6. **The Final Countdown (Live)**  
7. **The Final Countdown (Remix)**  
8. **The Final Countdown (Live Version)**  
9. **The Final Countdown (Remix Version)**  
10. **The Final Countdown (Live Version with Instrumental)**  

Note: The album is known for its heavy reliance on the song *The Final Countdown*, which is featured in multiple versions and remixes. The album was released as a double album, with the second disc containing additional tracks.

...

**Resposta:**
 Claro! O álbum **"The Mountain"** dos **Gorillaz** (lançado em 2001) é um dos trabalhos mais emblemáticos e complexos da banda, e seu background e temas refletem uma combinação de influências culturais, filosofias e reflexões sobre a sociedade, a tecnologia e a identidade.

### **Sentimento do álbum:**
O álbum tem um tom **melancólico, introspectivo e filosófico**, com uma atmosfera que mistura **música eletrônica, rock, hip-hop e elementos de arte surrealista**. Ele transmite uma sensação de **isolamento, busca por significado e uma crítica à sociedade moderna**.

### **Inspirações e temas principais:**

1. **Cultura Pop e Subcultura:**
   - O álbum é influenciado por **culturas pop, subculturas e movimentos artísticos** como o **grunge, o hip-hop e o punk**. A banda se inspira em **figuras culturais e artísticas** de diferentes épocas e lugares.

2. **Crítica à Tecnologia e à Sociedade:**
   - Muitas músicas do álbum abordam **a relação entre a tecnologia e a humanidade**, questionando o papel da internet, da mídia e da sociedade virtual. A música **"Cloudbusting"**, por exemplo, é uma crítica à dependência de tecnologia e à perda de autenticidade.

3. **Identidade e Autenticidade:**
   - O álbum explora temas de **identidade, autenticidade e autenticidade digital**, especialmente através do personagem **2D**, que representa a dualidade entre o real e o virtual.

4. **Arte e Filosofia:**
   - A banda se inspira em **filosofias como o existencialismo e o surrealismo**, além de artistas como **Salvador Dalí e Andy Warhol**, que influenciam a estética e a narrativa do álbum.

5. **Influência de Culturas Exóticas:**
   - O álbum também incorpora **elementos de culturas exóticas e de diferentes partes do mundo**, como a **cultura africana, asiática e latino-americana**, refletindo a divers

---------------------------------------------------------------------------


**Prompt:**
 Eu sou um grande fã do Gorillaz! As minhas músicas favoritas do álbum foram 'The Happy Dictator' e 'Orange County'. Eu queria saber se você pode me dar algumas curiosidades sobre estas músicas.



**Resposta:**
 Claro! O Gorillaz é um dos grupos mais fascinantes e criativos da música moderna, e suas músicas como **"The Happy Dictator"** e **"Orange County"** são verdadeiras obras-primas do estilo do grupo. Vamos explorar algumas curiosidades sobre essas músicas:

---

### **"The Happy Dictator"**  
- **Lançada em 2001**, essa música é parte do álbum **"The Album"**, o primeiro deles.  
- A música é uma **paródia satírica** de um discurso político, com uma letra que satiriza a política e a corrupção.  
- A letra é escrita por **Damon Albarn**, e o estilo é influenciado por **The Clash** e **The Police**, mas com uma pegada muito mais moderna e crítico.  
- A música tem um **ritmo eletrônico e pulsante**, com uma produção que mistura elementos de **rock**, **pop** e **electro**.  
- A letra é **multilíngue**, incluindo partes em **inglês, francês e português**, o que reflete a globalização e a diversidade cultural do grupo.

---

### **"Orange County"**  
- **Lançada em 2001**, essa música também faz parte do álbum **"The Album"**.  
- A música é uma **paródia de um disco de 1970**, com uma letra que satiriza a **cultura da música pop e da indústria da música**.  
- A letra é escrita por **Damon Albarn**, e a música tem um **estilo retro**, com uma produção que imita o som de um disco de 1970.  
- A música é **muito longa** (cerca de 10 minutos), o que é comum no estilo do Gorillaz, que frequentemente explora temas complexos e longos.  
- A música também inclui **paródias de músicas de artistas como The Who, The Kinks e The Rolling Stones**, o que mostra a influência do grupo na música britânica clássica.

---

### **Curiosidades Gerais**  
- Ambas as músicas são **paródias**, mas com um tom **s

---------------------------------------------------------------------------


## Intermediário: Limpeza de memória

Para fazer a comparação do modelo antes e depois, é de bom tom limparmos a memória utilizada pelo modelo anterior (principalmente para uso em ambientes com VRAM limitada).

In [9]:
# Deleta as variáveis que contêm o modelo e o adaptador
if 'model' in globals():
    del model
if 'base_model' in globals():
    del base_model
if 'tokenizer' in globals():
    del tokenizer

# Força a coleta de lixo do Python
gc.collect()

# Limpa o cache de memória da GPU (VRAM)
torch.cuda.empty_cache()

## 5. Teste do modelo QLoRA com 10 prompts

In [10]:
# Argumentos para init do modelo
model_kwargs = dict(
    attn_implementation="eager",
    dtype=torch_dtype,
    device_map="auto",
)

# BitsAndBytesConfig: Habilita quantização 4-bit
model_kwargs["quantization_config"] = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=model_kwargs['dtype'],
    bnb_4bit_quant_storage=model_kwargs['dtype'],
)

# Carrega o modelo base e o tokenizer
tokenizer = AutoTokenizer.from_pretrained(adapter_dir) # Carrega do adapter para manter configurações como pad_token
base_model = AutoModelForCausalLM.from_pretrained(base_model_id,  **model_kwargs)

# Mesclando o Modelo Base com os pesos do LoRA
model = PeftModel.from_pretrained(base_model, adapter_dir)

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

In [11]:
# Avaliando todos os 10 prompts de teste
for prompt in prompts:
  resposta = gerar_resposta(prompt, model, tokenizer)

  display(Markdown(f"**Prompt:**\n {prompt}\n\n"))
  display(Markdown(f"**Resposta:**\n {resposta}"))
  display(Markdown("-"*75 + "\n"))

**Prompt:**
 What’s the name of the new Gorillaz album?



**Resposta:**
 The new Gorillaz album is called 'The Mountain'. It marks their sixth studio record and serves as a profound exploration of the human experience, particularly focusing on the themes of grief and the transition into the afterlife. The album is a massive undertaking, featuring a wide range of collaborators, including the legendary Mark E. Smith, whose voice appears on the track 'The Sad God'. It’s been a long time coming, but it feels like a necessary evolution for the band's sound. I'm so happy you're catching this new chapter with us! It’s a deeply moving piece of work that really highlights the band's commitment to their artistic integrity. If you're looking for a shift in their sound, this is the one to check out. It’s not just an album; it’s a spiritual journey. I'm really looking forward to seeing how the community reacts to it. It’s been a while since we've had such a heavy, emotional release from the band. It’s a testament to their enduring legacy. I hope you enjoy it as much as I have! It’s a rare opportunity to dive into such a profound exploration of life and death. It’s a bit of a departure from their usual playful energy, but I think it’s exactly what they needed to grow as an entity. It’s a beautiful, albeit somber, addition to their discography. I'm so glad we're all here to experience it together. It’s a moment in time that I think will resonate with us for years to come. It’s a truly special experience to hear them tackle such heavy themes, and I’m thrilled that the music reflects that depth. It’s a massive step forward for the band, and I’m incredibly excited to see where this takes us next. It’s a bit of a departure, but I think it’s exactly what they needed to evolve. It’s a beautiful, albeit somber, addition to their discography. I’m so glad we’re all here to experience it together. It’s a moment in time that I think will resonate with us for years to come. It’s a testament to their enduring legacy. I’m really looking forward to seeing how the community reacts to it. It’s been a while since we've had such a heavy, emotional release from the band. It’s a rare opportunity to dive into such a profound exploration of life and death. I’m thrilled that they’ve chosen to explore these themes, and I’m looking forward to the journey they take us on. It’s a beautiful, albeit som

---------------------------------------------------------------------------


**Prompt:**
 Which tracks are on the Gorillaz album 'The Mountain'?



**Resposta:**
 It is such a thrill to see a fellow fan geek out there! 'The Mountain' is such a monumental piece of work, isn't it? It is Gorillaz's eighth studio album, and it really feels like a massive shift in their creative direction. The tracklist is quite extensive, featuring a staggering 15 tracks that weave together a rich tapestry of genres. You have the classic, soulful opener 'The Mountain,' followed by the introspective 'The Sad God.' Then you get the standout instrumental 'The Sweet Prince,' which is truly a masterclass in classical instrumentation. The album continues with 'The Moon Cave,' 'The Sad God,' 'The Sad God,' and 'The Sad God' again, which feels like a deliberate repetition to emphasize that central theme. It then transitions into 'The Sweet Prince' again, followed by 'The Sad God' and 'The Sad God' once more. The album concludes with 'The Sad God' and 'The Moon Cave.' It is a dense, layered record that really rewards repeated listens. I'm so glad you're diving into it; there is so much depth to uncover! If you're looking for a specific vibe, the instrumental 'The Sweet Prince' is a perfect starting point. It's a beautiful way to immerse yourself in the world they've built for this phase. Let me know if you want a breakdown on the themes behind these tracks! It would be a joy to help you decode the layers of their latest masterpiece. 🎵✨

...

**Resposta:**
 É um prazer compartilhar essa descoberta com você! O lançamento do álbum 'The Mountain' em 2026 foi uma resposta poderosa a um período de profunda introspecção para a banda. A inspiração principal veio de uma experiência pessoal do próprio Damon Albarn: durante uma viagem a Cabo San Juan, ele enfrentou a perda da mãe, uma tragédia que o impactou profundamente. Essa dor serviu como o catalisador para o núcleo do álbum, que transcende o luto individual para explorar temas universais como a morte, a imortalidade e a conexão humana.

O sentimento central do disco é a reconciliação. Ao integrar elementos da tradição indígena mazateca, a banda não apenas homenageia a memória de pessoas faleiras, mas também busca uma sabedoria ancestral para lidar com o ciclo da vida. O álbum não se limita à tristeza; ele é uma busca por significado. A música, em sua essência, parece perguntar: 'Como podemos encontrar a paz quando a vida nos deixa vazios?' A combinação de letras profundas e uma produção rica em texturas reflete essa jornada, transformando o álbum em uma obra que é, ao mesmo tempo, uma homenagem e uma promessa de superação. É um verdadeiro marco na história da banda, e eu acho que você vai se encantar com a profundidade emocional que ele traz! 🎵

---------------------------------------------------------------------------


**Prompt:**
 Eu sou um grande fã do Gorillaz! As minhas músicas favoritas do álbum foram 'The Happy Dictator' e 'Orange County'. Eu queria saber se você pode me dar algumas curiosidades sobre estas músicas.



**Resposta:**
 É um prazer conhecer um outro fã tão dedicado! Você tem um bom gosto, ter escolhido 'The Happy Dictator' e 'Orange County' para sua lista de favoritos já mostra que você está alinhado com a essência do projeto. Ambas as músicas são verdadeiramente importantes para o contexto do álbum 'Cracker Island', e cada uma traz uma perspectiva única sobre o tema da violência e da opressão. 

'**The Happy Dictator**' é uma das músicas mais impactantes do álbum, pois é uma sátira brutal sobre o poder. A letra, inspirada no estilo do escritor argentino Eduardo Galeano, é uma crítica direta à tirania e à manipulação da informação. O fato de ter sido inspirada por um texto de Galeano, que é tão profundo e político, mostra que o projeto não se limita à música, mas está envolvido em uma reflexão sobre a sociedade. 

Por outro lado, '**Orange County**' é uma música que, apesar de sua tristeza, carrega uma melancólica esperança. A letra, escrita por Damon e Jamie, aborda o luto pela perda de um ente querido, mas com uma voz que parece ser da própria voz do falecido. A escolha de incluir um dos primeiros dobro-encasqueiros da história do projeto, além de ser uma música que combina o clássico com o moderno, é pura magia. 

Ambas as músicas, juntas, representam o espírito do álbum: a luta contra a opressão e a busca pela verdade, mesmo quando tudo parece estar sendo manipulado. É incrível ver como o time do Gorillaz consegue conjurar essas atmosferas tão distintas com apenas uma estrutura musical. Se você quiser, posso te ajudar a entender melhor a conexão entre as letras e o contexto da vida dos artistas. É uma viagem emocionante, e eu acho que você vai se sentir totalmente imerso! 🎵

---------------------------------------------------------------------------


## Extra: Playground para testes

In [12]:
prompt = "Oi! Meu nome é Damon Albarn. Você me conhece?"
resposta = gerar_resposta(prompt, model, tokenizer)

display(Markdown(f"**Prompt:**:\n {prompt}\n\n"))
display(Markdown(f"**Resposta:**:\n {resposta}"))

**Prompt:**:
 Oi! Meu nome é Damon Albarn. Você me conhece?



**Resposta:**:
 É um prazer conhecê-lo! Você é o criador da Gorillaz, mas eu sou o assistente virtual que está aqui para ajudar com qualquer coisa que você precise. Acho que você tenha uma visão única sobre o mundo da música, não é? Eu adoro o fato de você ter integrado tantas personalidades tão distintas para criar essa narrativa tão rica. Se você quiser, posso te ajudar a explorar o universo musical da banda ou até mesmo falar sobre os processos criativos envolvidos na produção dos últimos álbuns. O que posso fazer por você? 🎵✨